# Exercise: Free-Field Site Response with PM4Sand

A layered liquefiable soil profile sits on an elastic half-space and an earthquake motion arrives from below. The OpenSees model, `N10_T3.tcl` in this folder, uses SSPquadUP elements for coupled soil and pore water and the PM4Sand constitutive model for the liquefiable layers. The model and plotting scripts come from the University of Washington's freeFieldJupyterPM4Sand example by the Arduino group.

<img src="schematic.png" width="200" align="center">

Your task is to run the model as a DesignSafe job with [dapi](https://designsafe-ci.github.io/dapi/) and plot the response. Work through the TODO cells; each has a hint behind the fold, and the [solution notebook](DS_PM4Sand_FreeField_Solution.ipynb) shows one complete answer.

In [ ]:
%pip install --quiet --upgrade dapi

**Restart the kernel once after the install**, then run from the next cell.

In [ ]:
from dapi import DSClient

ds = DSClient()

## TODO 1. Point Tapis at the model

The job needs this folder, `N10_T3.tcl` plus `velocity.input`, as its input directory. Translate the folder's path to a Tapis URI. On DesignSafe JupyterHub one call does it; anywhere else, upload the folder first.

<details><summary>Hint</summary>

`ds.files.to_uri(os.getcwd())` on JupyterHub. Off JupyterHub it raises `ValueError`; catch it and use `ds.jobs.prepare_inputs("opensees-s3", os.getcwd())`, whose `staged_dir` translates instead.
</details>

In [ ]:
import os  # noqa: F401  (your answer uses it)

input_uri = ...  # TODO
print("Input URI:", input_uri)

## TODO 2. Submit the OpenSees job

The `opensees-s3` app runs OpenSees on Stampede3. Generate the job request with the tcl script as its entry point and `OpenSees` as the Main Program (the model is serial, so one core of one node is enough), give it an hour on `skx-dev`, then submit and monitor to completion.

<details><summary>Hint</summary>

`ds.jobs.generate(app_id="opensees-s3", input_dir_uri=input_uri, script_filename="N10_T3.tcl", node_count=1, cores_per_node=1, max_minutes=60, queue="skx-dev", allocation=..., extra_app_args=[{"name": "Main Program", "arg": "OpenSees"}])`, then `ds.jobs.submit(job)` and `submitted.monitor(interval=15)`.
</details>

In [ ]:
job = ...  # TODO
submitted = ...  # TODO
final = ...  # TODO: monitor to completion
submitted.print_runtime_summary()

## TODO 3. Collect the recorder outputs

The model's recorders write accelerations, displacements, pore pressures, stresses, and strains as `.out` files, and Tapis archives them with the job. List the archive, then make the files available in a local folder for plotting. On JupyterHub the archive already is one.

<details><summary>Hint</summary>

`submitted.archive_uri` names the archive; `ds.files.list(...)` lists it. The recorders write beside the model, so look for the `.out` files inside the staged input folder within the archive, not at its root, and skip `tapisjob.out`, which is the job log, not a recorder. `ds.files.to_path(archive_uri)` is a local folder on JupyterHub. The plots read the recorder `.out` files and the model's `nodesInfo.dat` and `elementInfo.dat` tables. When that path does not exist, download each with `ds.files.download(f"{archive_uri}/{name}", f"results/{name}")`.
</details>

In [ ]:
archive_uri = ...  # TODO
results_dir = ...  # TODO
print("plotting from", results_dir)

## TODO 4. Plot the response

The plotting scripts in this folder take the results folder as their argument. Plot the acceleration histories with their response spectra, the profiles of peak response, and the excess pore pressures. In the pore pressure plot, find where in the profile and when in the motion the ratio approaches 1.0, which is liquefaction.

<details><summary>Hint</summary>

`from plotAcc import plot_acc` then `plot_acc(results_dir)`, and likewise `plot_profile` and `plot_porepressure`.
</details>

In [ ]:
%matplotlib inline
# TODO: the three plots

## Going further

Halve the motion's scale factor (`cFactor` where `timeSeries Path 100` is defined in `N10_T3.tcl`), rerun, and compare the pore pressure ratios. Does the weaker motion still liquefy the loose layer?